# 08 — Baseline Models

This notebook establishes baseline supervised-learning performance for severe-rainfall forecasting from +1h to +6h.

Three models are evaluated:

- Dummy Classifier;
- Logistic Regression;
- Random Forest.

The models are trained using observations from 2018–2020 and evaluated exclusively on 2021 data.

Because severe rainfall represents less than 1% of the available observations, overall accuracy is not considered an appropriate primary metric.

Evaluation focuses on:

- precision;
- recall;
- F1-score;
- average precision (PR-AUC);
- ROC-AUC;
- confusion matrix components.

No hyperparameter tuning, resampling or decision-threshold optimization is performed at this stage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone

from sklearn.dummy import DummyClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

INPUT_PATH = Path(
    "../data/processed/recife_2018_2021_model_dataset.csv"
)

OUTPUT_PATH = Path(
    "../data/processed/baseline_model_results.csv"
)

RANDOM_STATE = 42

FORECAST_HORIZONS = [
    1,
    2,
    3,
    4,
    5,
    6,
]


# ============================================================
# 2. FINAL FEATURE SET
# ============================================================

FEATURE_COLUMNS = [

    # Current meteorological state
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",

    # Temporal variables
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",

    # Precipitation history
    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    # Meteorological history
    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",

    # Short-term changes
    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 3. LOAD MODEL DATASET
# ============================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_PATH}"
    )


df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime_utc"],
)


df = (
    df
    .set_index("datetime_utc")
    .sort_index()
)


print("=== MODEL DATASET LOADED ===")
print()

print(
    "Rows:",
    len(df)
)

print(
    "Features:",
    len(FEATURE_COLUMNS)
)

print(
    "Training rows:",
    int((df["split"] == "train").sum())
)

print(
    "Test rows:",
    int((df["split"] == "test").sum())
)


# ============================================================
# 4. DEFINE BASELINE MODELS
# ============================================================

models = {

    "dummy": DummyClassifier(
        strategy="most_frequent",
    ),

    "logistic_regression": Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


# ============================================================
# 5. TRAIN AND EVALUATE EACH HORIZON
# ============================================================

results = []


for horizon in FORECAST_HORIZONS:

    print()
    print("=" * 60)
    print(
        f"FORECAST HORIZON: +{horizon}h"
    )
    print("=" * 60)


    target_column = (
        f"target_{horizon}h"
    )

    eligibility_column = (
        f"eligible_final_{horizon}h"
    )


    # --------------------------------------------------------
    # Train dataset
    # --------------------------------------------------------

    train_data = df[
        (df["split"] == "train")
        &
        df[eligibility_column]
    ].copy()


    # --------------------------------------------------------
    # Test dataset
    # --------------------------------------------------------

    test_data = df[
        (df["split"] == "test")
        &
        df[eligibility_column]
    ].copy()


    X_train = (
        train_data[
            FEATURE_COLUMNS
        ]
    )

    y_train = (
        train_data[
            target_column
        ]
        .astype(int)
    )


    X_test = (
        test_data[
            FEATURE_COLUMNS
        ]
    )

    y_test = (
        test_data[
            target_column
        ]
        .astype(int)
    )


    print(
        "Train samples:",
        len(X_train),
        "| positives:",
        int(y_train.sum())
    )

    print(
        "Test samples:",
        len(X_test),
        "| positives:",
        int(y_test.sum())
    )


    # --------------------------------------------------------
    # Train each model
    # --------------------------------------------------------

    for model_name, model_template in models.items():

        model = clone(
            model_template
        )


        model.fit(
            X_train,
            y_train,
        )


        y_pred = model.predict(
            X_test
        )


        # ----------------------------------------------------
        # Probability / score for ranking metrics
        # ----------------------------------------------------

        if hasattr(
            model,
            "predict_proba",
        ):

            y_score = (
                model.predict_proba(
                    X_test
                )[:, 1]
            )

        elif hasattr(
            model,
            "decision_function",
        ):

            y_score = (
                model.decision_function(
                    X_test
                )
            )

        else:

            y_score = (
                y_pred.astype(float)
            )


        # ----------------------------------------------------
        # Confusion matrix
        # ----------------------------------------------------

        tn, fp, fn, tp = (
            confusion_matrix(
                y_test,
                y_pred,
                labels=[0, 1],
            )
            .ravel()
        )


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            y_pred,
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0,
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0,
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0,
        )

        average_precision = (
            average_precision_score(
                y_test,
                y_score,
            )
        )

        roc_auc = (
            roc_auc_score(
                y_test,
                y_score,
            )
        )


        results.append(
            {
                "horizon_hours":
                    horizon,

                "model":
                    model_name,

                "train_samples":
                    len(X_train),

                "train_positives":
                    int(y_train.sum()),

                "test_samples":
                    len(X_test),

                "test_positives":
                    int(y_test.sum()),

                "accuracy":
                    accuracy,

                "precision":
                    precision,

                "recall":
                    recall,

                "f1":
                    f1,

                "average_precision":
                    average_precision,

                "roc_auc":
                    roc_auc,

                "true_negatives":
                    int(tn),

                "false_positives":
                    int(fp),

                "false_negatives":
                    int(fn),

                "true_positives":
                    int(tp),
            }
        )


        print()
        print(
            model_name
        )

        print(
            f"Precision: {precision:.3f}"
        )

        print(
            f"Recall:    {recall:.3f}"
        )

        print(
            f"F1:        {f1:.3f}"
        )

        print(
            f"PR-AUC:    {average_precision:.3f}"
        )

        print(
            f"ROC-AUC:   {roc_auc:.3f}"
        )

        print(
            "Confusion matrix:",
            f"TN={tn}",
            f"FP={fp}",
            f"FN={fn}",
            f"TP={tp}",
        )


# ============================================================
# 6. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)


print()
print(
    "=== BASELINE MODEL RESULTS ==="
)


display(
    results_df[
        [
            "horizon_hours",
            "model",
            "precision",
            "recall",
            "f1",
            "average_precision",
            "roc_auc",
            "true_positives",
            "false_positives",
            "false_negatives",
        ]
    ].round(3)
)


# ============================================================
# 7. F1 COMPARISON
# ============================================================

f1_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="f1",
    )
)


print()
print(
    "=== F1 SCORE BY HORIZON ==="
)

display(
    f1_comparison.round(3)
)


# ============================================================
# 8. RECALL COMPARISON
# ============================================================

recall_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="recall",
    )
)


print()
print(
    "=== RECALL BY HORIZON ==="
)

display(
    recall_comparison.round(3)
)


# ============================================================
# 9. AVERAGE PRECISION COMPARISON
# ============================================================

ap_comparison = (
    results_df
    .pivot(
        index="horizon_hours",
        columns="model",
        values="average_precision",
    )
)


print()
print(
    "=== AVERAGE PRECISION BY HORIZON ==="
)

display(
    ap_comparison.round(3)
)


# ============================================================
# 10. SAVE RESULTS
# ============================================================

results_df.to_csv(
    OUTPUT_PATH,
    index=False,
)


print()
print(
    "Baseline results saved to:"
)

print(
    OUTPUT_PATH
)

=== MODEL DATASET LOADED ===

Rows: 33871
Features: 20
Training rows: 26304
Test rows: 7567

FORECAST HORIZON: +1h
Train samples: 25586 | positives: 70
Test samples: 7248 | positives: 38

dummy
Precision: 0.000
Recall:    0.000
F1:        0.000
PR-AUC:    0.005
ROC-AUC:   0.500
Confusion matrix: TN=7210 FP=0 FN=38 TP=0

logistic_regression
Precision: 0.016
Recall:    0.947
F1:        0.031
PR-AUC:    0.206
ROC-AUC:   0.925
Confusion matrix: TN=4947 FP=2263 FN=2 TP=36

random_forest
Precision: 0.000
Recall:    0.000
F1:        0.000
PR-AUC:    0.116
ROC-AUC:   0.891
Confusion matrix: TN=7210 FP=0 FN=38 TP=0

FORECAST HORIZON: +2h
Train samples: 25571 | positives: 70
Test samples: 7246 | positives: 38

dummy
Precision: 0.000
Recall:    0.000
F1:        0.000
PR-AUC:    0.005
ROC-AUC:   0.500
Confusion matrix: TN=7208 FP=0 FN=38 TP=0

logistic_regression
Precision: 0.014
Recall:    0.921
F1:        0.027
PR-AUC:    0.078
ROC-AUC:   0.888
Confusion matrix: TN=4656 FP=2552 FN=3 TP=35

rando

,horizon_hours,model,precision,recall,f1,average_precision,roc_auc,true_positives,false_positives,false_negatives
0,1,dummy,0.000,0.000,0.000,0.005,0.500,0,0,38
1,1,logistic_regression,0.016,0.947,0.031,0.206,0.925,36,2263,2
2,1,random_forest,0.000,0.000,0.000,0.116,0.891,0,0,38
3,2,dummy,0.000,0.000,0.000,0.005,0.500,0,0,38
4,2,logistic_regression,0.014,0.921,0.027,0.078,0.888,35,2552,3
5,2,random_forest,0.000,0.000,0.000,0.048,0.742,0,0,38
6,3,dummy,0.000,0.000,0.000,0.005,0.500,0,0,38
7,3,logistic_regression,0.014,0.974,0.027,0.064,0.881,37,2624,1
8,3,random_forest,0.000,0.000,0.000,0.023,0.748,0,0,38
9,4,dummy,0.000,0.000,0.000,0.005,0.500,0,0,38



=== F1 SCORE BY HORIZON ===


model,dummy,logistic_regression,random_forest
horizon_hours,,,
1,0.0,0.031,0.0
2,0.0,0.027,0.0
3,0.0,0.027,0.0
4,0.0,0.026,0.0
5,0.0,0.023,0.0
6,0.0,0.022,0.0



=== RECALL BY HORIZON ===


model,dummy,logistic_regression,random_forest
horizon_hours,,,
1,0.0,0.947,0.0
2,0.0,0.921,0.0
3,0.0,0.974,0.0
4,0.0,0.921,0.0
5,0.0,0.868,0.0
6,0.0,0.816,0.0



=== AVERAGE PRECISION BY HORIZON ===


model,dummy,logistic_regression,random_forest
horizon_hours,,,
1,0.005,0.206,0.116
2,0.005,0.078,0.048
3,0.005,0.064,0.023
4,0.005,0.068,0.017
5,0.005,0.044,0.009
6,0.005,0.041,0.006



Baseline results saved to:
..\data\processed\baseline_model_results.csv


## Baseline Conclusion

The baseline experiments confirmed the difficulty of severe-rainfall forecasting under extreme class imbalance.

The Dummy Classifier predicted no severe-rainfall events and therefore achieved zero recall and F1-score.

Logistic Regression detected most severe-rainfall observations, achieving high recall across all forecasting horizons. However, the default classification threshold produced a large number of false positives and consequently very low precision.

Random Forest produced useful probability rankings, particularly for shorter forecasting horizons, but its default probability threshold of 0.5 resulted in no positive predictions.

The strongest baseline discrimination was observed for the +1h horizon, where Logistic Regression achieved an ROC-AUC of 0.925 and an average precision of 0.206.

These results indicate that probability-threshold selection is necessary before evaluating the models as final classifiers.

To avoid using the 2021 test period for model selection, subsequent threshold optimization will use a chronological validation period within the training data.